# Phase 8 — Machine learning models

This self-contained notebook tests whether engineered rolling-form features and
off-the-shelf classifiers improve on the evaluated statistical baselines (Elo,
Poisson) from Phase 7. Features come from `src/matchcast/features/rolling.py`
(leakage-safe recent goals for/against, win rate, opponent-adjusted form, neutral
venue, tournament type, and Elo rating trend); evaluation reuses the same
`src/matchcast/evaluation/` split and metric code and the identical 2014/2018/2022
expanding-window World Cup folds as Phase 7, so results are directly comparable.

**Seed:** `20260705` everywhere a model or split needs one.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

from matchcast.evaluation.baseline import TrainingFrequencyBaseline
from matchcast.evaluation.metrics import accuracy, multiclass_brier_score, multiclass_log_loss
from matchcast.evaluation.splits import chronological_split, expanding_window_tournament_folds
from matchcast.features.rolling import build_match_features

SEED = 20260705
DATA = _ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
REPORTS = _ROOT / 'reports'
TOURNAMENT = 'FIFA World Cup'
BACKTEST_YEARS = [2014, 2018, 2022]
LABEL_MAP = {'H': 0, 'D': 1, 'A': 2}

matches = pd.read_csv(DATA, parse_dates=['date'])
featured = build_match_features(matches, window=5)
featured.shape

(49493, 33)

## Feature matrix

Predictors: pre-match Elo difference, neutral venue, each side's rolling goals
for/against, win rate, opponent-adjusted form (points earned minus Elo-expected
points), Elo rating trend, and a one-hot tournament-type bucket.

In [2]:
NUMERIC_FEATURES = [
    'elo_difference', 'neutral',
    'home_rolling_goals_for', 'home_rolling_goals_against', 'home_rolling_win_rate',
    'home_rolling_form_vs_expected', 'home_rolling_elo_trend',
    'away_rolling_goals_for', 'away_rolling_goals_against', 'away_rolling_win_rate',
    'away_rolling_form_vs_expected', 'away_rolling_elo_trend',
]
TOURNAMENT_DUMMIES = ['world_cup', 'qualifier', 'continental', 'friendly']  # 'other' is the reference level


def design_matrix(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[NUMERIC_FEATURES].astype(float).copy()
    out['neutral'] = frame['neutral'].astype(float)
    for level in TOURNAMENT_DUMMIES:
        out[f'tournament_{level}'] = (frame['tournament_type'] == level).astype(float)
    return out


def labels_of(frame: pd.DataFrame) -> np.ndarray:
    return frame['result'].map(LABEL_MAP).to_numpy()


assert design_matrix(featured.head(3)).shape[1] == len(NUMERIC_FEATURES) + len(TOURNAMENT_DUMMIES)
design_matrix(featured).describe().T[['mean', 'std', 'min', 'max']].round(2)

,mean,std,min,max
elo_difference,6.91,163.08,-861.48,831.03
neutral,0.27,0.44,0.00,1.00
home_rolling_goals_for,1.48,0.90,0.00,17.00
home_rolling_goals_against,1.45,1.01,0.00,24.00
home_rolling_win_rate,0.50,0.24,0.00,1.00
home_rolling_form_vs_expected,0.00,0.19,-0.69,0.69
home_rolling_elo_trend,0.16,17.43,-75.65,63.50
away_rolling_goals_for,1.45,0.89,0.00,21.00
away_rolling_goals_against,1.49,1.06,0.00,21.00
away_rolling_win_rate,0.50,0.24,0.00,1.00


## Chronological hyperparameter tuning

A chronological 70/15/15 train/validation/test split (restricted to 2000-01-01
onward, matching the Phase 5 Poisson training window) selects hyperparameters using
validation-set log loss. The held-out test slice here is only used to sanity-check
the tuned models before the Phase 7-style backtest below; it is never used to choose
a hyperparameter.

In [3]:
modern = featured.loc[featured['date'] >= '2000-01-01'].sort_values(['date', 'match_id']).reset_index(drop=True)
tuning_split = chronological_split(modern, 'date', validation_fraction=0.15, test_fraction=0.15)
X_train, y_train = design_matrix(tuning_split.train), labels_of(tuning_split.train)
X_val, y_val = design_matrix(tuning_split.validation), labels_of(tuning_split.validation)
print(f'train={len(X_train):,} val={len(X_val):,} test={len(tuning_split.test):,}')

train=17,801 val=3,815 test=3,815


In [4]:
logistic_grid = [0.01, 0.1, 1.0, 10.0]
logistic_results = []
for c in logistic_grid:
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(C=c, max_iter=2000, random_state=SEED).fit(scaler.transform(X_train), y_train)
    val_probs = model.predict_proba(scaler.transform(X_val))
    logistic_results.append({'C': c, 'val_log_loss': multiclass_log_loss(
        np.array(['H', 'D', 'A'])[y_val], val_probs)})
logistic_results = pd.DataFrame(logistic_results)
best_logistic_c = float(logistic_results.loc[logistic_results['val_log_loss'].idxmin(), 'C'])
print('best logistic C =', best_logistic_c)
logistic_results

best logistic C = 0.1


,C,val_log_loss
0,0.01,0.859577
1,0.10,0.859332
2,1.00,0.859466
3,10.00,0.859482


In [5]:
forest_grid = [
    {'n_estimators': 200, 'max_depth': 4},
    {'n_estimators': 200, 'max_depth': 8},
    {'n_estimators': 400, 'max_depth': 4},
    {'n_estimators': 400, 'max_depth': None},
]
forest_results = []
for params in forest_grid:
    model = RandomForestClassifier(random_state=SEED, n_jobs=-1, **params).fit(X_train, y_train)
    val_probs = model.predict_proba(X_val)
    forest_results.append({**params, 'val_log_loss': multiclass_log_loss(
        np.array(['H', 'D', 'A'])[y_val], val_probs)})
forest_results = pd.DataFrame(forest_results)
best_forest_params = forest_grid[int(forest_results['val_log_loss'].idxmin())]
print('best random forest params =', best_forest_params)
forest_results

best random forest params = {'n_estimators': 200, 'max_depth': 8}


,n_estimators,max_depth,val_log_loss
0,200,4.0,0.910732
1,200,8.0,0.877202
2,400,4.0,0.911816
3,400,NaN,0.885636


## Calibration check

`CalibratedClassifierCV` (isotonic, 3-fold) is applied to the tuned random forest
only if it lowers validation log loss versus the uncalibrated model; the same check
is run for logistic regression, which is already a probabilistic model and is not
expected to need it.

In [6]:
def val_log_loss_for(estimator) -> float:
    probs = estimator.predict_proba(X_val)
    return multiclass_log_loss(np.array(['H', 'D', 'A'])[y_val], probs)


uncalibrated_forest = RandomForestClassifier(random_state=SEED, n_jobs=-1, **best_forest_params).fit(X_train, y_train)
calibrated_forest = CalibratedClassifierCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1, **best_forest_params), method='isotonic', cv=3
).fit(X_train, y_train)

uncalibrated_loss = val_log_loss_for(uncalibrated_forest)
calibrated_loss = val_log_loss_for(calibrated_forest)
use_calibrated_forest = calibrated_loss < uncalibrated_loss
print(f'uncalibrated={uncalibrated_loss:.4f} calibrated={calibrated_loss:.4f} '
      f'-> {"using calibrated" if use_calibrated_forest else "keeping uncalibrated"}')

uncalibrated=0.8772 calibrated=0.8779 -> keeping uncalibrated


## Gradient boosting candidate

Added only after the scikit-learn baselines above ran end to end, per the roadmap's
"simplest first" ordering. A small manual grid is tuned on the same chronological
validation split.

In [7]:
xgb_grid = [
    {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05},
    {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.03},
    {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05},
]
xgb_results = []
for params in xgb_grid:
    model = XGBClassifier(
        objective='multi:softprob', num_class=3, random_state=SEED,
        eval_metric='mlogloss', **params,
    ).fit(X_train, y_train)
    val_probs = model.predict_proba(X_val)
    xgb_results.append({**params, 'val_log_loss': multiclass_log_loss(
        np.array(['H', 'D', 'A'])[y_val], val_probs)})
xgb_results = pd.DataFrame(xgb_results)
best_xgb_params = xgb_grid[int(xgb_results['val_log_loss'].idxmin())]
print('best xgboost params =', best_xgb_params)
xgb_results

best xgboost params = {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.03}


,n_estimators,max_depth,learning_rate,val_log_loss
0,200,3,0.05,0.862123
1,300,3,0.03,0.861933
2,200,4,0.05,0.863521


## Head-to-head comparison on the Phase 7 harness

Every candidate is refit from scratch on each expanding-window World Cup fold's
train-only data (2014, 2018, 2022) and scored with the identical metrics used for
Elo and Poisson in Phase 7, so the comparison is apples-to-apples.

In [8]:
folds = expanding_window_tournament_folds(featured, 'date', 'tournament', TOURNAMENT, BACKTEST_YEARS)
assert [year for year, _, _ in folds] == BACKTEST_YEARS


def fit_predict(model_name: str, train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    X_tr, y_tr = design_matrix(train), labels_of(train)
    X_te = design_matrix(test)
    if model_name == 'baseline':
        return TrainingFrequencyBaseline.fit(train['result']).predict(len(test))
    if model_name == 'elo':
        return test[['elo_home_probability', 'elo_draw_probability', 'elo_away_probability']].to_numpy(float)
    if model_name == 'logistic':
        scaler = StandardScaler().fit(X_tr)
        model = LogisticRegression(C=best_logistic_c, max_iter=2000, random_state=SEED).fit(scaler.transform(X_tr), y_tr)
        return model.predict_proba(scaler.transform(X_te))
    if model_name == 'random_forest':
        base = RandomForestClassifier(random_state=SEED, n_jobs=-1, **best_forest_params)
        model = CalibratedClassifierCV(base, method='isotonic', cv=3).fit(X_tr, y_tr) if use_calibrated_forest else base.fit(X_tr, y_tr)
        return model.predict_proba(X_te)
    if model_name == 'xgboost':
        model = XGBClassifier(
            objective='multi:softprob', num_class=3, random_state=SEED,
            eval_metric='mlogloss', **best_xgb_params,
        ).fit(X_tr, y_tr)
        return model.predict_proba(X_te)
    raise ValueError(model_name)


MODEL_NAMES = ['baseline', 'elo', 'logistic', 'random_forest', 'xgboost']
fold_rows, pooled_labels, pooled_probs = [], [], {name: [] for name in MODEL_NAMES}
for year, train, test in folds:
    labels = test['result'].to_numpy()
    row = {'year': year, 'n_matches': len(test)}
    for name in MODEL_NAMES:
        probs = fit_predict(name, train, test)
        pooled_probs[name].append(probs)
        row[f'{name}_log_loss'] = multiclass_log_loss(labels, probs)
        row[f'{name}_brier'] = multiclass_brier_score(labels, probs)
        row[f'{name}_accuracy'] = accuracy(labels, probs)
    fold_rows.append(row)
    pooled_labels.append(labels)

fold_table = pd.DataFrame(fold_rows)
fold_table

,year,n_matches,baseline_log_loss,baseline_brier,baseline_accuracy,elo_log_loss,elo_brier,elo_accuracy,logistic_log_loss,logistic_brier,logistic_accuracy,random_forest_log_loss,random_forest_brier,random_forest_accuracy,xgboost_log_loss,xgboost_brier,xgboost_accuracy
0,2014,64,1.059328,0.641554,0.453125,0.977957,0.584495,0.53125,0.941322,0.557509,0.593750,0.973786,0.576786,0.640625,0.982933,0.580356,0.656250
1,2018,64,1.094111,0.667756,0.390625,0.970941,0.579230,0.56250,0.955264,0.567474,0.578125,0.984682,0.587733,0.562500,0.969113,0.576619,0.546875
2,2022,64,1.074251,0.651156,0.437500,1.031078,0.607093,0.53125,1.049956,0.610105,0.531250,1.017545,0.604332,0.484375,1.039997,0.610343,0.484375


In [9]:
pooled_labels_arr = np.concatenate(pooled_labels)
comparison_rows = []
for name in MODEL_NAMES:
    probs = np.concatenate(pooled_probs[name])
    comparison_rows.append({
        'model': name,
        'log_loss': multiclass_log_loss(pooled_labels_arr, probs),
        'brier': multiclass_brier_score(pooled_labels_arr, probs),
        'accuracy': accuracy(pooled_labels_arr, probs),
        'log_loss_std_across_folds': fold_table[f'{name}_log_loss'].std(ddof=0),
    })
comparison = pd.DataFrame(comparison_rows).set_index('model')
comparison['n_matches'] = int(fold_table['n_matches'].sum())
comparison.sort_values('log_loss')

,log_loss,brier,accuracy,log_loss_std_across_folds,n_matches
model,,,,,
logistic,0.982180,0.578363,0.567708,0.048261,192
random_forest,0.992005,0.589617,0.562500,0.018600,192
elo,0.993325,0.590273,0.541667,0.026849,192
xgboost,0.997348,0.589106,0.562500,0.030681,192
baseline,1.075897,0.653489,0.427083,0.014248,192


## Embedded validation checks

Covers leakage boundaries, probability normalization, feature-matrix shape, and
reproducibility of the tuned models.

In [10]:
for year, train, test in folds:
    assert train['date'].max() < test['date'].min()
for name in MODEL_NAMES:
    probs = np.concatenate(pooled_probs[name])
    assert np.isfinite(probs).all() and (probs >= 0).all()
    assert np.allclose(probs.sum(axis=1), 1.0)

# Refitting the logistic model on the same fold with the same seed is deterministic.
_, first_train, first_test = folds[0]
repeat = fit_predict('logistic', first_train, first_test)
assert np.allclose(repeat, pooled_probs['logistic'][0])

assert design_matrix(featured).isna().sum().sum() == 0
assert comparison['log_loss'].min() > 0
print('All Phase 8 validation checks passed.')

All Phase 8 validation checks passed.


## Interpretation and limitations

Hyperparameters, seeds, and validation log losses used for tuning are printed above
each grid; the final head-to-head table reports pooled and per-fold performance on
the same 192-match World Cup backtest used in Phase 7. See
`reports/model_comparison.md` for the recorded decision, runtimes, and full
configuration. As in Phase 7, three folds of 64 matches each is a small sample, so
per-fold variation should not be over-read; feature engineering choices (a 5-match
rolling window, a coarse 5-bucket tournament type) were not themselves tuned against
this backtest to avoid overfitting the evaluation harness.